# Generative AI — Assignment 1
### Part 1: Topic Detection & Summarization of BBC News Articles &nbsp;|&nbsp; Part 2: Job Postings Analysis

**Submitted by:** Kushagra Gupta
**Due:** 14th September, 9 PM &nbsp;|&nbsp; **Total:** 100 marks (+20 bonus)

---

**Approach.**
- Required steps (Part 1 on the first 30 articles, Part 2 on the first 25 postings) run on **Groq**
  (`openai/gpt-oss-120b`) — fast, and well within free-tier limits at this scale.
- The **Bonus** sections (full dataset — all 2,225 articles / all 2,277 postings) run on **Ollama**
  with a small local model, exactly as the assignment recommends, to avoid Groq rate limits.
  Ollama needs to be installed and running locally (`ollama serve`) with a model pulled
  (`ollama pull llama3.2`) — that part can't be executed inside this notebook-authoring
  environment, so it's written and documented for you to run on your own machine.
- Each classification/extraction task uses a **Pydantic schema + `PydanticOutputParser`**, so every
  LLM response is validated into a typed object instead of parsed by hand.

## Shared setup

Both parts reuse this Groq client. (The Ollama client for the bonus sections is set up separately,
right before it's first used, since it's optional.)

In [ ]:
%pip install -q langchain langchain-core langchain-groq langchain-ollama pandas pydantic

In [ ]:
import os, time, json, getpass
import pandas as pd
from typing import List, Optional
from typing_extensions import Literal
from pydantic import BaseModel, Field

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_groq import ChatGroq

# Groq free-tier key: https://console.groq.com/keys
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ_API_KEY: ")

# openai/gpt-oss-120b: Groq's current flagship, has strict structured-output support.
# (llama-3.3-70b-versatile was deprecated by Groq on 2026-08-16 — don't use it.)
# If you hit rate limits, switch to the lighter "openai/gpt-oss-20b".
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_retries=2)
print("Groq model ready:", llm.model_name)

---
# Part 1 — Topic Detection & Summarization of BBC News Articles (45 marks)

## Step 1 — Load the dataset

The BBC News Archive CSV is **tab-separated**, not comma-separated (plain `pd.read_csv` without
`sep="\t"` collapses every row into one column). Columns: `category`, `filename`, `title`, `content`.

In [ ]:
BBC_PATH = "bbc-news-data.csv"

bbc_full = pd.read_csv(BBC_PATH, sep="\t")
bbc = bbc_full.head(30).copy().reset_index(drop=True)

print("Full dataset:", bbc_full.shape, "| Working subset:", bbc.shape)
print("Categories in full set:\n", bbc_full["category"].value_counts().to_string())
bbc.head(3)

In [ ]:
# Note: the Kaggle file is grouped by category, so a literal head(30) is all "business"
# (see the value_counts above). The assignment asks specifically for df.head(30), so that's
# what's used below — but it means the sample demos in Steps 2-4 will mostly show "Business"
# as the detected topic. The full-dataset bonus run later naturally covers all 5 categories.
print(bbc["category"].value_counts().to_string())

## Step 2 — Topic Classification (10 marks)

Single-label classification over the 5 known categories, with two short few-shot examples to anchor
the model's labeling.

In [ ]:
class TopicResult(BaseModel):
    """Predicted topic for a news article."""
    topic: Literal["Business", "Entertainment", "Politics", "Sport", "Tech"] = Field(
        ..., description="The single best-fitting category for this article.")

topic_parser = PydanticOutputParser(pydantic_object=TopicResult)

TOPIC_FEWSHOT = """Example 1:
Article: "Shares in the tech firm rallied after it beat quarterly earnings forecasts, with revenue up 12% year-on-year."
Topic: Tech

Example 2:
Article: "The striker scored a last-minute winner to send his team through to the semi-final of the cup."
Topic: Sport
"""

topic_prompt = PromptTemplate(
    template=(
        "Analyze the following news article and identify its topic as one of these categories: "
        "Business, Entertainment, Politics, Sport, or Tech.\n\n"
        "{fewshot}\n"
        "Now classify this article:\n"
        "Title: {title}\n"
        "Article: {content}\n\n"
        "{format_instructions}"
    ),
    input_variables=["title", "content"],
    partial_variables={
        "fewshot": TOPIC_FEWSHOT,
        "format_instructions": topic_parser.get_format_instructions(),
    },
)

topic_chain = topic_prompt | llm | topic_parser
print("Topic classification chain ready.")

### Demo on a sample article

In [ ]:
sample = bbc.iloc[0]
topic_demo = topic_chain.invoke({"title": sample["title"], "content": sample["content"]})
print("Title  :", sample["title"])
print("Topic  :", topic_demo.topic)

## Step 3 — Summarization (10 marks)

Plain text output (no schema needed here) — 2-3 sentences, factual, no added commentary.

In [ ]:
summary_prompt = PromptTemplate(
    template=(
        "Summarize the main points of the following news article in 2-3 sentences. "
        "Cover the who/what/when/where/why where applicable. Do not add opinions or commentary "
        "that isn't in the article.\n\n"
        "Title: {title}\nArticle: {content}\n\nSummary:"
    ),
    input_variables=["title", "content"],
)

summary_chain = summary_prompt | llm | StrOutputParser()
print("Summarization chain ready.")

### Demo on a sample article

In [ ]:
summary_demo = summary_chain.invoke({"title": sample["title"], "content": sample["content"]})
print(summary_demo.strip())

## Step 4 — Key Entity Extraction (10 marks)

Entities are extracted into three typed buckets (people / organizations / locations) — more useful
internally than one flat blob — then flattened into a single list for the final `Key_Entities`
column, matching the assignment's example output format.

In [ ]:
class EntityResult(BaseModel):
    """Named entities mentioned in a news article."""
    people: List[str] = Field(default_factory=list, description="Notable people named in the article.")
    organizations: List[str] = Field(default_factory=list, description="Companies, institutions, teams, etc.")
    locations: List[str] = Field(default_factory=list, description="Places/countries/cities mentioned.")

entity_parser = PydanticOutputParser(pydantic_object=EntityResult)

entity_prompt = PromptTemplate(
    template=(
        "From the article below, list the important people, organizations, and places mentioned. "
        "Only include entities that actually appear in the text; use an empty list for any category "
        "with none.\n\n"
        "Title: {title}\nArticle: {content}\n\n{format_instructions}"
    ),
    input_variables=["title", "content"],
    partial_variables={"format_instructions": entity_parser.get_format_instructions()},
)

entity_chain = entity_prompt | llm | entity_parser
print("Entity extraction chain ready.")

### Demo on a sample article

In [ ]:
entity_demo = entity_chain.invoke({"title": sample["title"], "content": sample["content"]})
flat_entities_demo = entity_demo.people + entity_demo.organizations + entity_demo.locations
print(entity_demo.model_dump_json(indent=2))
print("\nFlattened Key_Entities:", flat_entities_demo)

## Step 5 — Apply to all 30 articles & update the DataFrame (15 marks)

Steps 2-4 were shown as three separate calls to demonstrate each task in isolation, as the brief
asks. For actually processing every row, doing 3 separate LLM calls per article (90 calls total)
is wasteful — so here they're merged into **one combined schema** and **one call per article**,
using `chain.batch()` so the 30 calls run concurrently instead of a serial loop.

In [ ]:
class ArticleAnalysis(BaseModel):
    """Combined topic + summary + entities for one article (single-call version of Steps 2-4)."""
    topic: Literal["Business", "Entertainment", "Politics", "Sport", "Tech"] = Field(
        ..., description="The single best-fitting category.")
    summary: str = Field(..., description="A factual 2-3 sentence summary of the article.")
    people: List[str] = Field(default_factory=list)
    organizations: List[str] = Field(default_factory=list)
    locations: List[str] = Field(default_factory=list)

article_parser = PydanticOutputParser(pydantic_object=ArticleAnalysis)

article_prompt = PromptTemplate(
    template=(
        "Analyze the news article below and return:\n"
        "1. topic - one of Business, Entertainment, Politics, Sport, Tech\n"
        "2. summary - 2-3 factual sentences covering who/what/when/where/why as applicable, "
        "no personal commentary\n"
        "3. people / organizations / locations - notable entities actually named in the article "
        "(empty list if none)\n\n"
        "Title: {title}\nArticle: {content}\n\n{format_instructions}"
    ),
    input_variables=["title", "content"],
    partial_variables={"format_instructions": article_parser.get_format_instructions()},
)

article_chain = article_prompt | llm | article_parser
print("Combined per-article chain ready.")

In [ ]:
bbc_inputs = [{"title": r["title"], "content": r["content"]} for _, r in bbc.iterrows()]

bbc_raw_results = article_chain.batch(
    bbc_inputs, config={"max_concurrency": 4}, return_exceptions=True
)

bbc_results, bbc_failed = [], []
for i, res in enumerate(bbc_raw_results):
    if isinstance(res, Exception):
        bbc_failed.append((i, str(res)))
        bbc_results.append(ArticleAnalysis(topic="Business", summary="Not available (extraction failed)."))
    else:
        bbc_results.append(res)

print(f"Processed {len(bbc_results)} articles, {len(bbc_failed)} failed.")
for i, e in bbc_failed:
    print("  row", i, "->", e[:150])

In [ ]:
bbc["Detected_Topic"] = [r.topic for r in bbc_results]
bbc["Summary"] = [r.summary for r in bbc_results]
bbc["Key_Entities"] = [r.people + r.organizations + r.locations for r in bbc_results]

bbc_final = bbc.rename(columns={"content": "Article_Text", "title": "Title"})
bbc_final.insert(0, "Article_ID", bbc_final.index)

bbc_final[["Article_ID", "Title", "Detected_Topic", "Summary", "Key_Entities"]].head(10)

### One record as JSON (matches the assignment's example format)

In [ ]:
sample_row = bbc_final.iloc[0]
record = {
    "Article_ID": int(sample_row["Article_ID"]),
    "Title": sample_row["Title"],
    "Article_Text": sample_row["Article_Text"][:120] + "...",
    "Detected_Topic": sample_row["Detected_Topic"],
    "Summary": sample_row["Summary"],
    "Key_Entities": sample_row["Key_Entities"],
}
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
bbc_final.to_csv("bbc_news_analyzed_first30.csv", index=False)
print("Saved -> bbc_news_analyzed_first30.csv")
print("\nDetected_Topic distribution:")
print(bbc_final["Detected_Topic"].value_counts().to_string())

## Part 1 Bonus (optional, up to 10 marks) — all 2,225 articles via Ollama

The assignment specifically warns that running the *entire* dataset through Groq risks rate limits,
and recommends **Ollama** with a small local model instead. This section:

- sets up a local `ChatOllama` client (needs `ollama serve` running and a model pulled beforehand),
- reuses the exact same `ArticleAnalysis` schema and prompt from Step 5,
- processes the **full** 2,225-article dataset in chunks, saving a checkpoint CSV after each chunk
  so a long run can be resumed instead of restarted from scratch if it's interrupted.

**To run this locally:**
```bash
# once, in a terminal
ollama pull llama3.2        # ~3B params, good speed/quality tradeoff for this task
ollama serve                # if not already running
```
This section will not execute inside the notebook-authoring environment (no local Ollama server
available there) — it's written and ready to run on your own machine.

In [ ]:
from langchain_ollama import ChatOllama

# 3B-parameter model: fast enough for ~2200 sequential calls on a laptop CPU/GPU.
# Swap for "phi3.5" or "qwen2.5:3b" if you prefer a different SLM you already have pulled.
OLLAMA_MODEL = "llama3.2"

ollama_llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
ollama_article_chain = article_prompt | ollama_llm | article_parser
print("Ollama chain ready (model:", OLLAMA_MODEL, ") — requires `ollama serve` running locally.")

In [ ]:
import math

CHUNK_SIZE = 100                 # articles per checkpoint save
MAX_CONCURRENCY = 2              # small models on modest hardware: keep this low
CHECKPOINT_PATH = "bbc_full_analyzed_checkpoint.csv"

def analyze_full_bbc(df: pd.DataFrame, chain, checkpoint_path: str = CHECKPOINT_PATH):
    """Process every row of df through `chain`, resuming from an existing checkpoint if present."""
    if os.path.exists(checkpoint_path):
        done = pd.read_csv(checkpoint_path)
        start = len(done)
        print(f"Resuming from checkpoint: {start}/{len(df)} already done.")
    else:
        done = pd.DataFrame()
        start = 0

    n_chunks = math.ceil((len(df) - start) / CHUNK_SIZE)
    for c in range(n_chunks):
        lo = start + c * CHUNK_SIZE
        hi = min(lo + CHUNK_SIZE, len(df))
        chunk = df.iloc[lo:hi]
        inputs = [{"title": r["title"], "content": r["content"]} for _, r in chunk.iterrows()]

        raw = chain.batch(inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)
        rows = []
        for (_, r), res in zip(chunk.iterrows(), raw):
            if isinstance(res, Exception):
                rows.append({**r.to_dict(), "Detected_Topic": "Business",
                             "Summary": "Not available (extraction failed).", "Key_Entities": []})
            else:
                rows.append({**r.to_dict(), "Detected_Topic": res.topic, "Summary": res.summary,
                             "Key_Entities": res.people + res.organizations + res.locations})

        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        done.to_csv(checkpoint_path, index=False)
        print(f"  chunk {c+1}/{n_chunks} done -> {hi}/{len(df)} articles saved to {checkpoint_path}")

    return done

# Uncomment to actually run (2,225 articles — will take a while on a local SLM):
# bbc_full_result = analyze_full_bbc(bbc_full, ollama_article_chain)
# bbc_full_result.head()
print("Full-dataset runner defined. Uncomment the two lines above to execute it locally.")

---
# Part 2 — Job Postings Analysis: Role Categorization & Requirements Extraction (55 marks)

## Step 1 — Load the dataset

CSV columns: an unnamed index, `Job Title`, `Job Description`. Drop the stray index column.

In [ ]:
JOBS_PATH = "job_title_des.csv"

jobs_full = pd.read_csv(JOBS_PATH)
jobs_full = jobs_full.drop(columns=[c for c in jobs_full.columns if c.startswith("Unnamed")], errors="ignore")
jobs = jobs_full.head(25).copy().reset_index(drop=True)

print("Full dataset:", jobs_full.shape, "| Working subset:", jobs.shape)
print("Columns:", jobs.columns.tolist())
jobs.head(3)

## Step 2 — Job Category Classification (10 marks)

Uses the category list the assignment suggests. A short few-shot pair anchors the boundary cases
(e.g. a data-analyst role that could plausibly read as either Technology/IT or Finance).

In [ ]:
class JobCategoryResult(BaseModel):
    """Predicted domain category for a job posting."""
    category: Literal[
        "Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Others"
    ] = Field(..., description="The single best-fitting domain for this job posting.")

category_parser = PydanticOutputParser(pydantic_object=JobCategoryResult)

CATEGORY_FEWSHOT = """Example 1:
Job Title: Backend Developer
Description: "Build and maintain REST APIs in Python/Django, work with PostgreSQL, deploy on AWS."
Category: Technology/IT

Example 2:
Job Title: Financial Analyst
Description: "Prepare quarterly financial models, forecast revenue, and support budgeting for the finance team."
Category: Finance
"""

category_prompt = PromptTemplate(
    template=(
        "Given the following job title and description, categorize the job into one of these "
        "domains: Technology/IT, Finance, Marketing, Healthcare, Education, Others. "
        "If genuinely unsure, use Others.\n\n"
        "{fewshot}\n"
        "Now classify this posting:\n"
        "Job Title: {job_title}\nDescription: {job_description}\n\n"
        "{format_instructions}"
    ),
    input_variables=["job_title", "job_description"],
    partial_variables={
        "fewshot": CATEGORY_FEWSHOT,
        "format_instructions": category_parser.get_format_instructions(),
    },
)

category_chain = category_prompt | llm | category_parser
print("Category classification chain ready.")

### Demo on a sample posting

In [ ]:
job_sample = jobs.iloc[0]
category_demo = category_chain.invoke({
    "job_title": job_sample["Job Title"], "job_description": job_sample["Job Description"]
})
print("Title    :", job_sample["Job Title"])
print("Category :", category_demo.category)

## Step 3 — Requirements Extraction (30 marks)

One structured prompt pulls all three fields at once (skills / education / experience), as the
brief's "alternatively" option suggests, rather than three separate prompts. `Education_Required`
and `Experience_Required` default to `"Not specified"` when the posting doesn't mention them —
handled by the schema itself, not left to post-processing.

In [ ]:
class RequirementsResult(BaseModel):
    """Extracted requirements from a job description."""
    skills: List[str] = Field(
        default_factory=list,
        description="Specific skills, programming languages, tools, or domain knowledge mentioned."
    )
    education: str = Field(
        default="Not specified",
        description="Minimum education level required/preferred (e.g. 'Bachelor's degree'). "
                    "Use exactly 'Not specified' if the description doesn't mention one."
    )
    experience: str = Field(
        default="Not specified",
        description="Years of experience or experience level required (e.g. '3+ years'). "
                    "Use exactly 'Not specified' if the description doesn't mention one."
    )

requirements_parser = PydanticOutputParser(pydantic_object=RequirementsResult)

requirements_prompt = PromptTemplate(
    template=(
        "Extract the required skills, education level, and years of experience from the job "
        "description below. Only use information actually stated in the text — use "
        "'Not specified' for education or experience if the posting doesn't mention it.\n\n"
        "Job Title: {job_title}\nDescription: {job_description}\n\n{format_instructions}"
    ),
    input_variables=["job_title", "job_description"],
    partial_variables={"format_instructions": requirements_parser.get_format_instructions()},
)

requirements_chain = requirements_prompt | llm | requirements_parser
print("Requirements extraction chain ready.")

### Demo on a sample posting

In [ ]:
requirements_demo = requirements_chain.invoke({
    "job_title": job_sample["Job Title"], "job_description": job_sample["Job Description"]
})
print(requirements_demo.model_dump_json(indent=2))

## Step 4 — Apply the chain to all 25 postings (10 marks) & Step 5 — update the DataFrame (5 marks)

As in Part 1, classification and extraction are merged into **one schema and one call per
posting** for the actual batch run, then applied with `chain.batch()`.

In [ ]:
class JobAnalysis(BaseModel):
    """Combined category + requirements for one posting (single-call version of Steps 2-3)."""
    category: Literal[
        "Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Others"
    ] = Field(..., description="The single best-fitting domain.")
    skills: List[str] = Field(default_factory=list)
    education: str = Field(default="Not specified")
    experience: str = Field(default="Not specified")

job_parser = PydanticOutputParser(pydantic_object=JobAnalysis)

job_prompt = PromptTemplate(
    template=(
        "Analyze the job posting below and return:\n"
        "1. category - one of Technology/IT, Finance, Marketing, Healthcare, Education, Others\n"
        "2. skills - specific skills/technologies/tools mentioned (empty list if none)\n"
        "3. education - minimum education level required/preferred, or 'Not specified'\n"
        "4. experience - years/level of experience required, or 'Not specified'\n\n"
        "Job Title: {job_title}\nDescription: {job_description}\n\n{format_instructions}"
    ),
    input_variables=["job_title", "job_description"],
    partial_variables={"format_instructions": job_parser.get_format_instructions()},
)

job_chain = job_prompt | llm | job_parser
print("Combined per-posting chain ready.")

In [ ]:
job_inputs = [{"job_title": r["Job Title"], "job_description": r["Job Description"]}
              for _, r in jobs.iterrows()]

job_raw_results = job_chain.batch(job_inputs, config={"max_concurrency": 4}, return_exceptions=True)

job_results, job_failed = [], []
for i, res in enumerate(job_raw_results):
    if isinstance(res, Exception):
        job_failed.append((i, str(res)))
        job_results.append(JobAnalysis(category="Others"))
    else:
        job_results.append(res)

print(f"Processed {len(job_results)} postings, {len(job_failed)} failed.")
for i, e in job_failed:
    print("  row", i, "->", e[:150])

In [ ]:
# Retry only the rows that failed above, sequentially with backoff.
# The batch call above fires everything concurrently with no pacing, so a per-minute
# rate limit on `openai/gpt-oss-120b` can reject a burst of requests at once (429).
# Retrying one-at-a-time with real waits between attempts clears most of these without
# having to reprocess the rows that already succeeded.

def invoke_with_backoff(chain, payload, max_retries=5):
    last_error = None
    for attempt in range(max_retries):
        try:
            return chain.invoke(payload)
        except Exception as exc:
            last_error = exc
            if attempt < max_retries - 1:
                is_rate_limit = "429" in str(exc) or "rate_limit" in str(exc).lower()
                wait = 20 * (attempt + 1) if is_rate_limit else 2 ** attempt
                print(f"    retry {attempt + 1}/{max_retries - 1} after {wait}s ({exc}...)")
                time.sleep(wait)
    raise last_error


still_failed = []
for i, _ in job_failed:
    try:
        job_results[i] = invoke_with_backoff(job_chain, job_inputs[i])
        print(f"  row {i} -> recovered")
    except Exception as exc:
        still_failed.append((i, str(exc)))
        print(f"  row {i} -> still failing: {str(exc)[:150]}")
    time.sleep(1)  # small gap even between successful sequential calls

print(f"\nRetried {len(job_failed)} rows; {len(still_failed)} still failing.")
job_failed = still_failed

In [ ]:
jobs["Predicted_Category"] = [r.category for r in job_results]
jobs["Required_Skills"] = [r.skills for r in job_results]
jobs["Education_Required"] = [r.education for r in job_results]
jobs["Experience_Required"] = [r.experience for r in job_results]

jobs.head(10)

### One record as JSON (matches the assignment's example format)

In [ ]:
sample_job_row = jobs.iloc[0]
job_record = {
    "Job_Title": sample_job_row["Job Title"],
    "Job_Description": sample_job_row["Job Description"][:150] + "...",
    "Predicted_Category": sample_job_row["Predicted_Category"],
    "Required_Skills": sample_job_row["Required_Skills"],
    "Education_Required": sample_job_row["Education_Required"],
    "Experience_Required": sample_job_row["Experience_Required"],
}
print(json.dumps(job_record, indent=2, ensure_ascii=False))

In [ ]:
jobs.to_csv("job_postings_analyzed_first25.csv", index=False)
print("Saved -> job_postings_analyzed_first25.csv")
print("\nPredicted_Category distribution:")
print(jobs["Predicted_Category"].value_counts().to_string())

## Part 2 Bonus (optional, up to 10 marks) — all 2,277 postings via Ollama

Same pattern as the Part 1 bonus: reuse the Ollama client, same chunked-with-checkpoint runner,
same `JobAnalysis` schema. Also written to run locally, not inside this authoring environment.

In [ ]:
ollama_job_chain = job_prompt | ollama_llm | job_parser

JOBS_CHECKPOINT_PATH = "jobs_full_analyzed_checkpoint.csv"

def analyze_full_jobs(df: pd.DataFrame, chain, checkpoint_path: str = JOBS_CHECKPOINT_PATH):
    """Process every row of df through `chain`, resuming from an existing checkpoint if present."""
    if os.path.exists(checkpoint_path):
        done = pd.read_csv(checkpoint_path)
        start = len(done)
        print(f"Resuming from checkpoint: {start}/{len(df)} already done.")
    else:
        done = pd.DataFrame()
        start = 0

    n_chunks = math.ceil((len(df) - start) / CHUNK_SIZE)
    for c in range(n_chunks):
        lo = start + c * CHUNK_SIZE
        hi = min(lo + CHUNK_SIZE, len(df))
        chunk = df.iloc[lo:hi]
        inputs = [{"job_title": r["Job Title"], "job_description": r["Job Description"]}
                  for _, r in chunk.iterrows()]

        raw = chain.batch(inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)
        rows = []
        for (_, r), res in zip(chunk.iterrows(), raw):
            if isinstance(res, Exception):
                rows.append({**r.to_dict(), "Predicted_Category": "Others", "Required_Skills": [],
                             "Education_Required": "Not specified", "Experience_Required": "Not specified"})
            else:
                rows.append({**r.to_dict(), "Predicted_Category": res.category, "Required_Skills": res.skills,
                             "Education_Required": res.education, "Experience_Required": res.experience})

        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        done.to_csv(checkpoint_path, index=False)
        print(f"  chunk {c+1}/{n_chunks} done -> {hi}/{len(df)} postings saved to {checkpoint_path}")

    return done

# Uncomment to actually run (2,277 postings — will take a while on a local SLM):
# jobs_full_result = analyze_full_jobs(jobs_full, ollama_job_chain)
# jobs_full_result.head()
print("Full-dataset runner defined. Uncomment the two lines above to execute it locally.")

---
## Wrap-up

- **Part 1** (required): `bbc_news_analyzed_first30.csv` — 30 articles, `Detected_Topic` /
  `Summary` / `Key_Entities`.
- **Part 2** (required): `job_postings_analyzed_first25.csv` — 25 postings, `Predicted_Category` /
  `Required_Skills` / `Education_Required` / `Experience_Required`.
- **Bonus** (optional, run locally with Ollama): uncomment the two marked lines in each bonus
  section to process the full 2,225 / 2,277 rows. Both write resumable checkpoint CSVs, so an
  interrupted run picks up where it left off rather than starting over.